In [2]:
import torch
import torch.nn as nn

In [3]:
layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [4]:
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

In [5]:
def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)

model = nn.Sequential(nn.Linear(50, 40), nn.ReLU(), nn.Linear(40, 1), nn.ReLU())
model.apply(use_he_init)

Sequential(
  (0): Linear(in_features=50, out_features=40, bias=True)
  (1): ReLU()
  (2): Linear(in_features=40, out_features=1, bias=True)
  (3): ReLU()
)

In [6]:
alpha = 0.2
model = nn.Sequential(nn.Linear(50, 40), nn.LeakyReLU(negative_slope=alpha))
nn.init.kaiming_uniform_(model[0].weight, alpha, nonlinearity="leaky_relu")

Parameter containing:
tensor([[-0.2134, -0.3323,  0.0776,  ..., -0.1963, -0.1246, -0.2034],
        [ 0.3326, -0.1252, -0.2712,  ...,  0.0695, -0.2642, -0.1652],
        [-0.1477, -0.2645, -0.1162,  ..., -0.1184,  0.2051,  0.2965],
        ...,
        [-0.1713, -0.2826, -0.0118,  ...,  0.3102,  0.3384, -0.1767],
        [-0.0746,  0.3213,  0.1222,  ..., -0.1594,  0.0988,  0.1398],
        [-0.1999, -0.0363, -0.1524,  ...,  0.2133,  0.3336,  0.0063]],
       requires_grad=True)

In [7]:
model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(1 * 28 * 28),
    nn.Linear(1 * 28 * 28, 300),
    nn.ReLU(),
    nn.BatchNorm1d(300),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.BatchNorm1d(100),
    nn.Linear(100, 10),
)

In [8]:
dict(model[1].named_parameters()).keys()

dict_keys(['weight', 'bias'])

In [9]:
dict(model[1].named_buffers()).keys()

dict_keys(['running_mean', 'running_var', 'num_batches_tracked'])

In [10]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 300, bias=False),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Linear(300, 100, bias=False),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

In [11]:
inputs = torch.randn(32, 3, 100, 200)
layer_norm = nn.LayerNorm([100, 200])
result = layer_norm(inputs)

In [12]:
means = inputs.mean(dim=[2, 3], keepdim=True)
vars_ = inputs.var(dim=[2, 3], keepdim=True, unbiased=False)
stds = torch.sqrt(vars_ + layer_norm.eps)
result = layer_norm.weight * (inputs - means) / stds + layer_norm.bias

In [13]:
layer_norm = nn.LayerNorm([3, 100, 200])
result = layer_norm(inputs)

In [14]:
#Gradiemt clipping

# for epoch in range(n_epochs):
#     for X_batch, y_batch in train_loader:
#         X_batch, y_batch = X_batch.to(device), y_batch.to(device)
#         y_pred = model(X_batch)
#         loss = loss_fn(y_pred, y_batch)
#         loss.backward()
#         nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()
#         optimizer.zero_grad()

In [15]:
torch.manual_seed(42)

model_A = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 8),
)

# train this model or load pretrained weights

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [17]:
import copy

torch.manual_seed(42)
reused_layers = copy.deepcopy(model_A[:-1])
model_B_on_A = nn.Sequential(
    *reused_layers,
    nn.Linear(100, 1)
).to(device)

In [18]:
for layer in model_B_on_A[:-1]:
    for param in layer.parameters():
        param.requires_grad = False

In [19]:
import torchmetrics

xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
# train model_B_on_A

In [20]:
#Momentum
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.05)

In [21]:
#Nesterov
optimizer = torch.optim.SGD(model.parameters(),
                            momentum=0.9, nesterov=True, lr=0.05)

In [22]:
#RMSProp
optimizer = torch.optim.RMSprop(model.parameters(), alpha=0.9, lr=0.05)

In [23]:
#Adam
optimizer = torch.optim.Adam(model.parameters(), betas=(0.9, 0.999), lr=0.05)

In [24]:
#AdamW
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

In [25]:
#Some model
# model = nn.Sequential(nn.Linear(10, 10))
#
# optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
# scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

In [26]:
# for epoch in range(n_epochs):
#     for X_batch, y_batch in train_loader:
#         [...] the rest of the training loop remains unchanged
#
#     scheduler.step()

In [27]:
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=20, eta_min=0.001)

In [28]:
# [...]  build the model and optimizer
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#   optimizer, mode="max", patience=2, factor=0.1)

In [29]:
# metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
# for epoch in range(n_epochs):
#     for X_batch, y_batch in train_loader:
#         [...]  the rest of the training loop remains unchanged
#     val_metric = evaluate_tm(model, valid_loader, metric).item()
#     scheduler.step(val_metric)

In [30]:
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=3)

In [31]:
warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda epoch: (min(epoch, 3) / 3) * (1.0 - 0.1) + 0.1)

In [32]:
# for epoch in range(n_epochs):
#     warmup_scheduler.step()
#     for X_batch, y_batch in train_loader:
#         [...]   the rest of the training loop is unchanged
#     if epoch >= 3:   deactivate other scheduler(s) during warmup
#         scheduler.step(val_metric)

In [1]:
# cosine_repeat_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     optimizer, T_0=2, T_mult=2, eta_min=0.001)

In [33]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, weight_decay=1e-4)
# use the optimizer normally during training

In [35]:
# n_epochs = 100
# optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
# params_to_regularize = [
#     param for name, param in model.named_parameters()
#     if not "bias" in name and not "bn" in name
# ]
# for epoch in range(n_epochs):
#     for X_batch, y_batch in train_loader:
#         [...]  # the rest of the training loop is unchanged
#         main_loss = loss_fn(y_pred, y_batch)
#         l2_loss = sum(param.pow(2.0).sum() for param in params_to_regularize)
#         loss = main_loss + 1e-4 * l2_loss
#         [...]

In [36]:
# l1_loss = sum(param.abs().sum() for param in params_to_regularize)
# loss = main_loss + 1e-4 * l1_loss

In [37]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Dropout(p=0.2), nn.Linear(1 * 28 * 28, 100), nn.ReLU(),
    nn.Dropout(p=0.2), nn.Linear(100, 100), nn.ReLU(),
    nn.Dropout(p=0.2), nn.Linear(100, 100), nn.ReLU(),
    nn.Dropout(p=0.2), nn.Linear(100, 10)
).to(device)

In [ ]:
model.eval()
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()

X_new = [...]  # some new images, e.g., the first 3 images of the test set
X_new = X_new.to(device)

torch.manual_seed(42)
with torch.no_grad():
    X_new_repeated = X_new.repeat_interleave(100, dim=0)
    y_logits_all = model(X_new_repeated).reshape(3, 100, 10)
    y_probas_all = torch.nn.functional.softmax(y_logits_all, dim=-1)
    y_probas = y_probas_all.mean(dim=1)

In [ ]:
y_probas.round(decimals=2)

In [ ]:
y_std = y_probas_all.std(dim=1)
y_std.round(decimals=2)

In [40]:
import torch.nn.functional as F

class McDropout(nn.Dropout):
    def forward(self, input):
        return F.dropout(input, self.p, training=True)

In [ ]:
def apply_max_norm(model, max_norm=2, epsilon=1e-8, dim=1):
    with torch.no_grad():
        for name, param in model.named_parameters():
            if 'bias' not in name:
                actual_norm = param.norm(p=2, dim=dim, keepdim=True)
                target_norm = torch.clamp(actual_norm, 0, max_norm)
                param *= target_norm / (epsilon + actual_norm)